# VSDS Active Residue Entropy Analysis (MCA)

This notebook rewrites the old LSAE-based entropy analysis for the MCA protein embedding used by the current KinSPEC model.
It maps SHAP-important protein MCA components back to KLIFS residue bits using the MCA category coordinate/contribution files.


In [1]:
#!/usr/bin/env python
# coding: utf-8
"""
VSDS_truedecoy active set: MCA protein important-residue entropy analysis.

Per active (SMILES, UniProt_ID) pair:
  1. mol_fp -> molecule SAE -> selected molecular latent vector (466d)
  2. UniProt_ID -> KLIFS row -> MCA protein embedding (200d) + KLIFS onehot bits
  3. SHAP on XGBoost([mol_emb, prot_mca_emb]) -> top protein MCA components
  4. MCA category loading matrix -> 1955 KLIFS bit scores -> 85 residue scores
  5. top-N nonzero residues enter entropy enrichment analysis

The protein inverse mapping follows KinSPEC_interpretability_MCA_2.ipynb:
  - load MCA_200D_category_coordinates.csv
  - load MCA_200D_category_contributions.csv
  - load MCA_200D_category_lookup.csv
  - construct bit x component loading as sign(coordinate) * contribution

Outputs in OUTDIR:
  - per_pair_top10_residues.csv
  - position_frequency.csv
  - region_frequency.csv
  - significance.txt
  - region_entropy_vs_count.png
  - position_entropy_vs_count.png
  - entropy_distribution.png
"""

import os
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
import shap
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from scipy import stats
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x


# =========================
# Project/path helpers
# =========================
def find_project_root() -> Path:
    """Find the KinSEPC github root from either repo root or notebook directory."""
    anchors = [Path.cwd(), *Path.cwd().parents, Path("/home/s1856122/KinSEPC_upload/github")]
    for cand in anchors:
        if (
            (cand / "dataset/vsds-vd_truedecoy/VSDS_truedecoy_integrated_data.csv").exists()
            and (cand / "LSAE_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv").exists()
            and (cand / "model/KinSPEC_full_training_pool_MCA_200D.json").exists()
        ):
            return cand.resolve()
    raise FileNotFoundError("Could not locate KinSEPC github project root from current directory.")


PROJECT_ROOT = find_project_root()
print(f"[init] project root = {PROJECT_ROOT}", flush=True)


# =========================
# Paths
# =========================
VSDS_CSV = PROJECT_ROOT / "dataset/vsds-vd_truedecoy/VSDS_truedecoy_integrated_data.csv"

KLIFS_FILE = PROJECT_ROOT / "LSAE_embedding/Kinase_test_MCA/human_482_typical_kinase_1955bit.csv"
KLIFS_MCA_CSV = PROJECT_ROOT / "LSAE_embedding/Kinase_test_MCA/MCA_klifs_482_200D_features.csv"
KLIFS_ONEHOT = PROJECT_ROOT / "LSAE_embedding/kinase/human_482_typical_kinase_1955bit_onehot.csv"

MCA_CATEGORY_COORDS = PROJECT_ROOT / "LSAE_embedding/Kinase_test_MCA/MCA_200D_category_coordinates.csv"
MCA_CATEGORY_CONTRIB = PROJECT_ROOT / "LSAE_embedding/Kinase_test_MCA/MCA_200D_category_contributions.csv"
MCA_CATEGORY_LOOKUP = PROJECT_ROOT / "LSAE_embedding/Kinase_test_MCA/MCA_200D_category_lookup.csv"

MOL_SAE_PATH = PROJECT_ROOT / "LSAE_embedding/mol/LSAE_1900k_chembl_model_2.pt"
MOL_SAE_IDX = PROJECT_ROOT / "LSAE_embedding/mol/LSAE_chembl_1900k_topk_of1024d_top99_2.pt"

XGB_MODEL = PROJECT_ROOT / "model/KinSPEC_full_training_pool_MCA_200D.json"

OUTDIR = PROJECT_ROOT / "model_interpretability/vsds_active_residue_analysis_MCA"


# =========================
# Hyperparameters
# =========================
RADIUS = 2
NBITS = 4096
MOL_KEEP_DIMS = 466
TOPK_PROT_COMPONENTS = 5
TOPN_RESIDUES = 10
BACKGROUND_SIZE = 50
BITS_PER_RESIDUE = 23
N_RESIDUES = 85
N_BASELINE_SAMPLES = 10000
RANDOM_SEED = 42
MAX_ACTIVE_PAIRS = None  # set to a small integer for quick smoke tests
BITSCORE_METHOD = "normalized_with_z"  # "normalized_with_z", "normalized_no_z", or "simple_abs"

os.makedirs(OUTDIR, exist_ok=True)


# =========================
# KLIFS region map and entropy values
# =========================
KLIFS_DOMAIN_MAP = {}
for i in range(0, 3):    KLIFS_DOMAIN_MAP[i] = "I"
for i in range(3, 9):    KLIFS_DOMAIN_MAP[i] = "g.l."
for i in range(9, 13):   KLIFS_DOMAIN_MAP[i] = "II"
for i in range(13, 19):  KLIFS_DOMAIN_MAP[i] = "III"
for i in range(19, 30):  KLIFS_DOMAIN_MAP[i] = "alphaC"
for i in range(30, 37):  KLIFS_DOMAIN_MAP[i] = "b.l"
for i in range(37, 41):  KLIFS_DOMAIN_MAP[i] = "IV"
for i in range(41, 44):  KLIFS_DOMAIN_MAP[i] = "V"
KLIFS_DOMAIN_MAP[44] = "GK"
for i in range(45, 48):  KLIFS_DOMAIN_MAP[i] = "hinge"
for i in range(48, 52):  KLIFS_DOMAIN_MAP[i] = "Linker"
for i in range(52, 59):  KLIFS_DOMAIN_MAP[i] = "alphaD"
for i in range(59, 64):  KLIFS_DOMAIN_MAP[i] = "alphaE"
for i in range(64, 67):  KLIFS_DOMAIN_MAP[i] = "VI"
for i in range(67, 75):  KLIFS_DOMAIN_MAP[i] = "c.l"
for i in range(75, 78):  KLIFS_DOMAIN_MAP[i] = "VII"
KLIFS_DOMAIN_MAP[78] = "VIII"
for i in range(79, 83):  KLIFS_DOMAIN_MAP[i] = "xDFG"
for i in range(83, 85):  KLIFS_DOMAIN_MAP[i] = "a.l"

REGIONS_ORDERED = [
    "alphaD", "alphaC", "IV", "Linker", "I", "II", "alphaE", "b.l", "VI",
    "GK", "V", "a.l", "hinge", "III", "g.l.", "VII", "VIII", "c.l", "xDFG",
]
REGION_ENTROPY = {
    "alphaD": 2.133, "alphaC": 2.124, "IV": 2.016, "Linker": 1.958, "I": 1.944,
    "II": 1.881, "alphaE": 1.868, "b.l": 1.839, "VI": 1.827, "GK": 1.761,
    "V": 1.758, "a.l": 1.581, "hinge": 1.579, "III": 1.329, "g.l.": 1.280,
    "VII": 1.228, "VIII": 1.110, "c.l": 0.899, "xDFG": 0.811,
}
RES_ENTROPY = np.array(
    [REGION_ENTROPY[KLIFS_DOMAIN_MAP[i]] for i in range(N_RESIDUES)],
    dtype=np.float64,
)


# =========================
# SAE class for molecule encoder
# =========================
class SparseLinearAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim, latent_dim, bias=True)
        self.decoder = nn.Linear(latent_dim, input_dim, bias=True)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def find_col(df, candidates):
    """Case- and underscore-insensitive column resolver."""
    norm = {c.lower().replace("_", ""): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().replace("_", "")
        if key in norm:
            return norm[key]
    raise KeyError(f"None of {candidates} found in columns {list(df.columns)}")


def bitstring_to_array(bitstr: str) -> np.ndarray:
    bitstr = str(bitstr).strip()
    if len(bitstr) != N_RESIDUES * BITS_PER_RESIDUE:
        raise ValueError(f"Expected 1955 KLIFS bits, got {len(bitstr)}")
    return np.frombuffer(bitstr.encode("ascii"), dtype=np.uint8) - ord("0")


def clean_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Convert CSV cells such as '[3.8745418E-1]' to numeric floats."""
    cleaned = df.copy()
    for col in cleaned.columns:
        if cleaned[col].dtype == object:
            cleaned[col] = (
                cleaned[col]
                .astype(str)
                .str.strip()
                .str.replace(r"^\[\s*", "", regex=True)
                .str.replace(r"\s*\]$", "", regex=True)
            )
        cleaned[col] = pd.to_numeric(cleaned[col], errors="raise")
    return cleaned


def numeric_to_numpy(df: pd.DataFrame, dtype=np.float64) -> np.ndarray:
    return clean_numeric_frame(df).to_numpy(dtype=dtype)


def load_embedding_matrix(path: Path, csv_skiprows=1, csv_header=None) -> np.ndarray:
    ext = path.suffix.lower()
    if ext == ".csv":
        df = pd.read_csv(path, skiprows=csv_skiprows, header=csv_header)
        return numeric_to_numpy(df, dtype=np.float32)
    if ext == ".npy":
        return np.load(path).astype(np.float32)
    if ext == ".pt":
        obj = torch.load(path, map_location="cpu")
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().numpy().astype(np.float32)
        for key in ["emb", "embedding", "X", "features"]:
            if key in obj:
                return obj[key].detach().cpu().numpy().astype(np.float32)
        raise TypeError(f"Unsupported .pt content in {path}")
    raise ValueError(f"Unsupported embedding extension: {path}")


def load_klifs_onehot(klifs_df: pd.DataFrame) -> np.ndarray:
    if KLIFS_ONEHOT.exists():
        onehot = pd.read_csv(KLIFS_ONEHOT, header=None).values.astype(np.uint8)
        if onehot.shape == (len(klifs_df), N_RESIDUES * BITS_PER_RESIDUE):
            return onehot
        print(f"[warn] onehot shape {onehot.shape} does not match KLIFS rows; rebuilding from Binary_string", flush=True)
    bit_col = find_col(klifs_df, ["Binary_string", "binary_string", "KLIFS_bitstring"])
    return np.stack([bitstring_to_array(x) for x in klifs_df[bit_col]]).astype(np.uint8)


def normalise_mca_category_name(name):
    return str(name).replace("__", "_")


def load_mca_bit_loadings(prot_dim: int) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Build 1955 x prot_dim loading matrices from MCA category files.

    loadings = sign(category coordinate) * category contribution, matching the
    MCA inverse-mapping convention in KinSPEC_interpretability_MCA_2.ipynb.
    """
    component_cols = [f"mca_{i + 1}" for i in range(prot_dim)]
    coords_df = pd.read_csv(MCA_CATEGORY_COORDS)
    contrib_df = pd.read_csv(MCA_CATEGORY_CONTRIB)
    lookup_df = pd.read_csv(MCA_CATEGORY_LOOKUP)

    missing_coord_cols = [c for c in component_cols if c not in coords_df.columns]
    missing_contrib_cols = [c for c in component_cols if c not in contrib_df.columns]
    if missing_coord_cols or missing_contrib_cols:
        raise KeyError(
            "MCA loading files do not contain all protein embedding dimensions. "
            f"Missing coordinates: {missing_coord_cols[:5]}, "
            f"missing contributions: {missing_contrib_cols[:5]}"
        )

    coords_df = coords_df.copy()
    contrib_df = contrib_df.copy()
    coords_df["category_name_norm"] = coords_df.iloc[:, 0].map(normalise_mca_category_name)
    contrib_df["category_name_norm"] = contrib_df.iloc[:, 0].map(normalise_mca_category_name)

    coords_by_name = coords_df.set_index("category_name_norm")[component_cols]
    contrib_by_name = contrib_df.set_index("category_name_norm")[component_cols]

    coord_mat = np.zeros((N_RESIDUES * BITS_PER_RESIDUE, prot_dim), dtype=np.float64)
    contrib_mat = np.zeros_like(coord_mat)
    missing_categories = []

    for _, row in lookup_df.dropna(subset=["bit_index_0_based"]).iterrows():
        bit_idx = int(row["bit_index_0_based"])
        if bit_idx < 0 or bit_idx >= coord_mat.shape[0]:
            continue
        category_name = normalise_mca_category_name(row["category_name"])
        if category_name not in coords_by_name.index or category_name not in contrib_by_name.index:
            missing_categories.append(category_name)
            continue
        coord_mat[bit_idx] = numeric_to_numpy(coords_by_name.loc[[category_name]], dtype=np.float64)[0]
        contrib_mat[bit_idx] = numeric_to_numpy(contrib_by_name.loc[[category_name]], dtype=np.float64)[0]

    loadings = np.sign(coord_mat) * contrib_mat
    return loadings, contrib_mat, missing_categories


def sample_baseline_mean(rng, K, n_samples=N_BASELINE_SAMPLES):
    """Average mean entropy over random K-residue draws without replacement."""
    if K <= 0:
        return float("nan")
    keys = rng.random((n_samples, N_RESIDUES))
    idx = np.argpartition(keys, K - 1, axis=1)[:, :K]
    return float(RES_ENTROPY[idx].mean(axis=1).mean())


def extract_shap_vector(explainer, X_df: pd.DataFrame) -> np.ndarray:
    shap_values = explainer.shap_values(X_df)
    if isinstance(shap_values, list):
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    elif getattr(shap_values, "ndim", 0) == 3:
        shap_values = shap_values[:, :, 1] if shap_values.shape[2] > 1 else shap_values[:, :, 0]
    return np.array(shap_values).reshape(-1)


def mca_bitscore(prot_onehot, prot_emb, prot_shap, prot_loadings, top_components):
    bit_score = np.zeros(N_RESIDUES * BITS_PER_RESIDUE, dtype=np.float64)
    for comp_idx in top_components:
        if comp_idx < 0 or comp_idx >= prot_loadings.shape[1]:
            continue
        shap_k = prot_shap[comp_idx]
        z_k = prot_emb[comp_idx]
        W_k = prot_loadings[:, comp_idx]

        if BITSCORE_METHOD == "normalized_with_z":
            weighted = prot_onehot * np.abs(z_k) * np.abs(W_k)
            total = weighted.sum()
            if total > 1e-10:
                bit_score += np.abs(shap_k) * (weighted / total)
        elif BITSCORE_METHOD == "normalized_no_z":
            weighted = prot_onehot * np.abs(W_k)
            total = weighted.sum()
            if total > 1e-10:
                bit_score += np.abs(shap_k) * (weighted / total)
        elif BITSCORE_METHOD == "simple_abs":
            bit_score += prot_onehot * np.abs(shap_k) * np.abs(z_k) * np.abs(W_k)
        else:
            raise ValueError(f"Unknown BITSCORE_METHOD: {BITSCORE_METHOD}")
    return bit_score


# =========================
# Main
# =========================
def main():
    print("=" * 70, flush=True)
    print("VSDS active residue entropy analysis - MCA protein embedding", flush=True)
    print("=" * 70, flush=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[init] device = {device}", flush=True)
    print(f"[init] output = {OUTDIR}", flush=True)

    # --- molecule SAE ---
    mol_sae = SparseLinearAE(NBITS, 1024).to(device)
    mol_sae.load_state_dict(torch.load(MOL_SAE_PATH, map_location=device, weights_only=True))
    mol_sae.eval()
    mol_keep_idx = torch.load(MOL_SAE_IDX, map_location=device, weights_only=True).long()[:MOL_KEEP_DIMS]
    print(f"[init] molecule SAE latents kept = {len(mol_keep_idx)}", flush=True)

    # --- KLIFS lookup and MCA protein embedding ---
    klifs_df = pd.read_csv(KLIFS_FILE)
    uniprot_col = find_col(klifs_df, ["uniprot_id", "Uniprot_ID", "UniProt_ID"])
    print(f"[init] KLIFS uniprot column = '{uniprot_col}'", flush=True)

    uniprot_to_row = {}
    duplicate_uniprots = 0
    for row_idx, uid in enumerate(klifs_df[uniprot_col].astype(str).tolist()):
        if uid in uniprot_to_row:
            duplicate_uniprots += 1
        else:
            uniprot_to_row[uid] = row_idx
    if duplicate_uniprots:
        print(f"[warn] {duplicate_uniprots} duplicate UniProt IDs in KLIFS file; first row used", flush=True)

    klifs_mca_mat = load_embedding_matrix(KLIFS_MCA_CSV, csv_skiprows=1, csv_header=None)
    klifs_onehot_mat = load_klifs_onehot(klifs_df)
    prot_dim = klifs_mca_mat.shape[1]
    print(f"[init] KLIFS MCA shape = {klifs_mca_mat.shape}, onehot shape = {klifs_onehot_mat.shape}", flush=True)

    if klifs_mca_mat.shape[0] != len(klifs_df):
        print(f"[warn] row mismatch: MCA rows={klifs_mca_mat.shape[0]} vs KLIFS rows={len(klifs_df)}", flush=True)
    if klifs_onehot_mat.shape[0] != len(klifs_df):
        print(f"[warn] row mismatch: onehot rows={klifs_onehot_mat.shape[0]} vs KLIFS rows={len(klifs_df)}", flush=True)

    prot_loadings, prot_contrib, missing_mca_categories = load_mca_bit_loadings(prot_dim)
    print(f"[init] MCA bit loadings shape = {prot_loadings.shape}", flush=True)
    print(f"[init] MCA bit categories with contributions = {np.sum(np.any(prot_contrib > 0, axis=1))} / {prot_contrib.shape[0]}", flush=True)
    if missing_mca_categories:
        print(f"[warn] {len(missing_mca_categories)} MCA categories missing from coordinate/contribution files", flush=True)
        print(f"       examples: {missing_mca_categories[:10]}", flush=True)

    # --- XGBoost + SHAP ---
    booster = xgb.Booster()
    booster.load_model(str(XGB_MODEL))
    feat_cols = list(range(MOL_KEEP_DIMS + prot_dim))
    background = pd.DataFrame(
        np.zeros((BACKGROUND_SIZE, MOL_KEEP_DIMS + prot_dim), dtype=np.float32),
        columns=feat_cols,
    )
    explainer = shap.TreeExplainer(booster, data=background)
    print(f"[init] XGBoost + SHAP ready; feature dim = {MOL_KEEP_DIMS + prot_dim}", flush=True)

    # --- Morgan generator ---
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=NBITS)

    # --- VSDS ---
    vsds = pd.read_csv(VSDS_CSV)
    sm_col = find_col(vsds, ["smiles", "SMILES"])
    up_col = find_col(vsds, ["uniprot_id", "Uniprot_ID", "UniProt_ID"])
    lb_col = find_col(vsds, ["molecule_label", "label", "active"])
    labels = pd.to_numeric(vsds[lb_col], errors="coerce")
    actives = vsds[labels == 1].reset_index(drop=True)
    if MAX_ACTIVE_PAIRS is not None:
        actives = actives.head(int(MAX_ACTIVE_PAIRS)).reset_index(drop=True)
    print(f"[init] VSDS rows={len(vsds)} active rows={len(actives)}", flush=True)
    print(f"[init] columns smiles='{sm_col}' uniprot='{up_col}' label='{lb_col}'", flush=True)

    @torch.no_grad()
    def smiles_to_mol_emb(smiles):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        fp = morgan_gen.GetFingerprint(mol)
        arr = np.zeros((NBITS,), dtype=np.float32)
        arr[list(fp.GetOnBits())] = 1.0
        t = torch.from_numpy(arr).unsqueeze(0).to(device)
        z = mol_sae.encoder(t)[:, mol_keep_idx].cpu().numpy().squeeze()
        return z.astype(np.float32)

    # =========================
    # Per-pair processing
    # =========================
    mol_cache = {}
    prot_cache = {}

    position_count = np.zeros(N_RESIDUES, dtype=np.int64)
    per_pair_records = []
    obs_arr = []
    rand_arr = []

    rng = np.random.default_rng(RANDOM_SEED)

    skipped_smiles = 0
    skipped_uniprot = 0
    skipped_zerobitscore = 0

    print("\n[run ] processing active pairs ...", flush=True)
    for i in tqdm(range(len(actives)), file=sys.stdout):
        row = actives.iloc[i]
        smi = str(row[sm_col])
        uid = str(row[up_col])

        if smi not in mol_cache:
            mol_cache[smi] = smiles_to_mol_emb(smi)
        mol_emb = mol_cache[smi]
        if mol_emb is None:
            skipped_smiles += 1
            continue

        if uid not in prot_cache:
            if uid not in uniprot_to_row:
                prot_cache[uid] = (None, None)
            else:
                r = uniprot_to_row[uid]
                prot_cache[uid] = (
                    klifs_mca_mat[r].astype(np.float32),
                    klifs_onehot_mat[r].astype(np.uint8),
                )
        prot_emb, prot_onehot = prot_cache[uid]
        if prot_emb is None:
            skipped_uniprot += 1
            continue

        X = np.hstack([mol_emb, prot_emb]).reshape(1, -1).astype(np.float32)
        X_df = pd.DataFrame(X, columns=feat_cols)
        sv = extract_shap_vector(explainer, X_df)
        prot_shap = sv[MOL_KEEP_DIMS:]

        top_components = np.argsort(-np.abs(prot_shap))[:TOPK_PROT_COMPONENTS]
        bit_score = mca_bitscore(prot_onehot, prot_emb, prot_shap, prot_loadings, top_components)
        res_score = bit_score.reshape(N_RESIDUES, BITS_PER_RESIDUE).sum(axis=1)

        nonzero_idx = np.where(res_score > 0)[0]
        if len(nonzero_idx) == 0:
            skipped_zerobitscore += 1
            continue

        order = nonzero_idx[np.argsort(-res_score[nonzero_idx])]
        top_pos = order[:TOPN_RESIDUES]
        K = len(top_pos)

        for pos in top_pos:
            position_count[pos] += 1

        obs_mean_ent = float(RES_ENTROPY[top_pos].mean())
        rand_mean_ent = sample_baseline_mean(rng, K, n_samples=N_BASELINE_SAMPLES)

        obs_arr.append(obs_mean_ent)
        rand_arr.append(rand_mean_ent)

        per_pair_records.append({
            "smiles": smi,
            "uniprot_id": uid,
            "n_top_residues": K,
            "top_positions_0idx": ",".join(map(str, top_pos.tolist())),
            "top_positions_1idx": ",".join(map(str, (top_pos + 1).tolist())),
            "top_regions": ",".join(KLIFS_DOMAIN_MAP[p] for p in top_pos),
            "top_residue_scores": ",".join(f"{res_score[p]:.6g}" for p in top_pos),
            "top_mca_components_0idx": ",".join(map(str, top_components.tolist())),
            "top_mca_components_1idx": ",".join(map(str, (top_components + 1).tolist())),
            "top_mca_component_abs_shap": ",".join(f"{abs(prot_shap[c]):.6g}" for c in top_components),
            "observed_mean_entropy": obs_mean_ent,
            "random_mean_entropy": rand_mean_ent,
            "delta_entropy": obs_mean_ent - rand_mean_ent,
        })

    print(f"\n[run ] processed pairs   = {len(per_pair_records)}", flush=True)
    print(f"[run ] skipped (smiles)  = {skipped_smiles}", flush=True)
    print(f"[run ] skipped (uniprot) = {skipped_uniprot}", flush=True)
    print(f"[run ] skipped (zero bs) = {skipped_zerobitscore}", flush=True)

    if len(per_pair_records) == 0:
        raise RuntimeError("No successful active pairs were analysed.")

    # =========================
    # Aggregate tables
    # =========================
    df_pair = pd.DataFrame(per_pair_records)
    df_pair.to_csv(OUTDIR / "per_pair_top10_residues.csv", index=False)

    df_pos = pd.DataFrame({
        "position_0idx": np.arange(N_RESIDUES),
        "position_1idx": np.arange(1, N_RESIDUES + 1),
        "region": [KLIFS_DOMAIN_MAP[i] for i in range(N_RESIDUES)],
        "region_entropy": RES_ENTROPY,
        "count": position_count,
    })
    df_pos.to_csv(OUTDIR / "position_frequency.csv", index=False)

    region_count = defaultdict(int)
    region_n_pos = defaultdict(int)
    for pos in range(N_RESIDUES):
        region_count[KLIFS_DOMAIN_MAP[pos]] += int(position_count[pos])
        region_n_pos[KLIFS_DOMAIN_MAP[pos]] += 1

    df_region = pd.DataFrame({
        "region": REGIONS_ORDERED,
        "region_entropy": [REGION_ENTROPY[r] for r in REGIONS_ORDERED],
        "count": [region_count[r] for r in REGIONS_ORDERED],
        "n_positions": [region_n_pos[r] for r in REGIONS_ORDERED],
    })
    df_region.to_csv(OUTDIR / "region_frequency.csv", index=False)

    # =========================
    # Statistics
    # =========================
    obs = np.array(obs_arr, dtype=np.float64)
    ran = np.array(rand_arr, dtype=np.float64)
    delta = obs - ran

    try:
        w_two = stats.wilcoxon(obs, ran, alternative="two-sided", zero_method="wilcox")
        w_grt = stats.wilcoxon(obs, ran, alternative="greater", zero_method="wilcox")
        w_two_stat, w_two_p = float(w_two.statistic), float(w_two.pvalue)
        w_grt_stat, w_grt_p = float(w_grt.statistic), float(w_grt.pvalue)
    except ValueError as exc:
        print(f"[warn] Wilcoxon failed: {exc}", flush=True)
        w_two_stat = w_two_p = w_grt_stat = w_grt_p = float("nan")

    K_baseline_summary = (
        df_pair.groupby("n_top_residues")
        .agg(n=("random_mean_entropy", "size"), baseline=("random_mean_entropy", "mean"))
        .reset_index()
        .sort_values("n_top_residues")
    )

    sig_path = OUTDIR / "significance.txt"
    with open(sig_path, "w") as f:
        f.write("VSDS active residue entropy enrichment - MCA protein embedding\n")
        f.write(f"bitscore method = {BITSCORE_METHOD}\n")
        f.write(f"protein components per pair = top {TOPK_PROT_COMPONENTS} by |SHAP|\n")
        f.write(f"baseline = mean over {N_BASELINE_SAMPLES} random K-residue draws, per pair\n")
        f.write(f"n pairs analysed = {len(delta)}\n\n")

        f.write("Per-K averaged baselines\n")
        for _, row in K_baseline_summary.iterrows():
            K = int(row["n_top_residues"])
            f.write(f"  K = {K:>2}   n = {int(row['n']):>4}   baseline = {row['baseline']:.4f}\n")
        f.write("\n")

        f.write("Per-pair statistics\n")
        f.write(f"  observed mean entropy   {obs.mean():.4f}  +/-  {obs.std(ddof=1):.4f}\n")
        f.write(f"  baseline mean entropy   {ran.mean():.4f}  +/-  {ran.std(ddof=1):.6f}\n")
        f.write(f"  delta (obs - baseline)  {delta.mean():.4f}  +/-  {delta.std(ddof=1):.4f}\n\n")

        f.write("Wilcoxon signed-rank (paired)\n")
        f.write(f"  two-sided   stat = {w_two_stat:.4f}   p = {w_two_p:.3e}\n")
        f.write(f"  greater     stat = {w_grt_stat:.4f}   p = {w_grt_p:.3e}\n")

    with open(sig_path) as f:
        print("\n" + f.read(), flush=True)

    # =========================
    # Plots
    # =========================
    fig, ax1 = plt.subplots(figsize=(15, 7))
    x = np.arange(len(REGIONS_ORDERED))
    ax1.plot(
        x,
        [REGION_ENTROPY[r] for r in REGIONS_ORDERED],
        marker="o",
        linewidth=2.5,
        color="red",
        label="Mean residue entropy across 482 kinases",
    )
    ax1.set_xlabel("KLIFS functional region", fontsize=14)
    ax1.set_ylabel("Mean residue entropy", fontsize=14)
    ax1.set_xticks(x)
    ax1.set_xticklabels(REGIONS_ORDERED, rotation=45, ha="right")
    ax1.set_ylim(0.7, 2.3)

    ax2 = ax1.twinx()
    ax2.bar(
        x,
        df_region["count"].values,
        alpha=0.55,
        color="steelblue",
        label=f"VSDS active top-{TOPN_RESIDUES} residue counts (n={len(delta)} pairs)",
    )
    ax2.set_ylabel("Important-residue hit count", fontsize=14)

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right", frameon=True)
    ax1.set_title("KLIFS region entropy and VSDS active MCA important-residue distribution", fontsize=15)
    plt.tight_layout()
    plt.savefig(OUTDIR / "region_entropy_vs_count.png", dpi=300, bbox_inches="tight")
    plt.close()

    fig, ax1 = plt.subplots(figsize=(18, 7))
    x85 = np.arange(N_RESIDUES)
    ax1.step(x85, RES_ENTROPY, where="mid", color="red", linewidth=2, label="Region entropy assigned per residue")
    ax1.set_xlabel("KLIFS position (0-84)", fontsize=14)
    ax1.set_ylabel("Region entropy", fontsize=14)
    ax1.set_ylim(0.7, 2.3)

    ax2 = ax1.twinx()
    ax2.bar(x85, position_count, alpha=0.55, color="steelblue", label=f"VSDS active hit count (n={len(delta)} pairs)")
    ax2.set_ylabel("Hit count", fontsize=14)

    seg_start = 0
    for i in range(1, N_RESIDUES + 1):
        cur = KLIFS_DOMAIN_MAP[i] if i < N_RESIDUES else None
        prev = KLIFS_DOMAIN_MAP[i - 1]
        if cur != prev:
            if i < N_RESIDUES:
                ax1.axvline(i - 0.5, color="gray", linewidth=0.6, linestyle="--", alpha=0.5)
            ax1.text((seg_start + i - 1) / 2, 2.27, prev, ha="center", va="top", fontsize=8)
            seg_start = i

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=10)
    ax1.set_title("VSDS active MCA important residues at 85-position resolution", fontsize=14)
    plt.tight_layout()
    plt.savefig(OUTDIR / "position_entropy_vs_count.png", dpi=300, bbox_inches="tight")
    plt.close()

    fig, ax = plt.subplots(figsize=(9, 5.5))
    baseline_val = ran.mean()
    bins = np.linspace(min(obs.min(), baseline_val) - 0.02, obs.max() + 0.02, 50)
    ax.hist(obs, bins=bins, alpha=0.7, color="steelblue", edgecolor="white", linewidth=0.4,
            label="Observed (top important residues)")
    ax.axvline(baseline_val, color="dimgray", linestyle="--", linewidth=2,
               label=f"Random baseline {baseline_val:.3f}\n(mean of {N_BASELINE_SAMPLES} draws per pair)")
    ax.axvline(obs.mean(), color="steelblue", linestyle="--", linewidth=2, label=f"Observed mean {obs.mean():.3f}")
    ax.axvspan(baseline_val, bins[-1], alpha=0.08, color="steelblue")
    ax.set_xlabel("Mean entropy of K residues per pair")
    ax.set_ylabel("Pair count")
    ax.set_title(f"Observed entropy vs random baseline (Wilcoxon greater p = {w_grt_p:.2e}, n = {len(obs)} pairs)")
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.savefig(OUTDIR / "entropy_distribution.png", dpi=300, bbox_inches="tight")
    plt.close()

    print(f"\n[done] outputs written to {OUTDIR}/", flush=True)
    for fn in [
        "per_pair_top10_residues.csv",
        "position_frequency.csv",
        "region_frequency.csv",
        "significance.txt",
        "region_entropy_vs_count.png",
        "position_entropy_vs_count.png",
        "entropy_distribution.png",
    ]:
        print(f"       {fn}", flush=True)


if __name__ == "__main__":
    main()


/home/s1856122/anaconda3/envs/xgboost/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[init] project root = /home/s1856122/KinSEPC_upload/github
VSDS active residue entropy analysis - MCA protein embedding
[init] device = cuda
[init] output = /home/s1856122/KinSEPC_upload/github/model_interpretability/vsds_active_residue_analysis_MCA
[init] molecule SAE latents kept = 466
[init] KLIFS uniprot column = 'UniProt_ID'
[init] KLIFS MCA shape = (482, 200), onehot shape = (482, 1955)


/home/s1856122/anaconda3/envs/xgboost/lib/python3.12/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


[init] MCA bit loadings shape = (1955, 200)
[init] MCA bit categories with contributions = 1823 / 1955
[warn] 132 MCA categories missing from coordinate/contribution files
       examples: ['res03_sidechain_Imino', 'res03_sidechain_Acidic_sc', 'res03_functional_Acidic_fn', 'res03_size_Tiny', 'res04_sidechain_Aromatic', 'res06_sidechain_Aromatic', 'res06_sidechain_Imino', 'res08_sidechain_Imino', 'res11_sidechain_Acidic_sc', 'res11_functional_Acidic_fn']
[init] XGBoost + SHAP ready; feature dim = 666
[init] VSDS rows=49706 active rows=1198
[init] columns smiles='SMILES' uniprot='uniprot_id' label='molecule_label'

[run ] processing active pairs ...
100%|██████████| 1198/1198 [17:39<00:00,  1.13it/s]

[run ] processed pairs   = 1198
[run ] skipped (smiles)  = 0
[run ] skipped (uniprot) = 0
[run ] skipped (zero bs) = 0

VSDS active residue entropy enrichment - MCA protein embedding
bitscore method = normalized_with_z
protein components per pair = top 5 by |SHAP|
baseline = mean over 10000